In [ ]:
import ast
import re
import numpy as np
import pandas as pd
from collections import Counter
from scipy.stats import pearsonr
from skbio.stats.distance import DistanceMatrix, mantel

# ------------------
# Token Jaccard
# ------------------

def tokenize(text: str):
    return re.findall(r"[A-Za-z_]\w*|\d+|==|!=|<=|>=|[+\-*/%(){}\[\]]", text.lower())

def token_counter(code: str):
    return Counter(tokenize(code))

def multiset_jaccard_from_counters(c1, c2):
    keys = set(c1) | set(c2)
    inter = sum(min(c1[k], c2[k]) for k in keys)
    union = sum(max(c1[k], c2[k]) for k in keys)
    return inter / union if union > 0 else 0.0

# ------------------
# AST parse_ok（统一 gate）
# ------------------

def ast_parse_ok(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except Exception:
        return False

# ------------------
# 通用工具
# ------------------

def upper_triangle(mat):
    idx = np.triu_indices_from(mat, k=1)
    return mat[idx]

def matrix_pearson(D_sim, D_perf):
    return pearsonr(upper_triangle(D_sim), upper_triangle(D_perf))

def matrix_mantel(D_sim, D_perf, perms=999):
    dm1 = DistanceMatrix(D_sim.astype(np.float32))
    dm2 = DistanceMatrix(D_perf.astype(np.float32))
    r, p, _ = mantel(dm1, dm2, method="pearson", permutations=perms)
    return r, p

def overlap_at_k(D_sim, D_perf, k=10):
    n = len(D_sim)
    vals = []
    for i in range(n):
        ds = D_sim[i].copy()
        dp = D_perf[i].copy()
        ds[i] = dp[i] = np.inf
        nn_s = np.argpartition(ds, k)[:k]
        nn_p = np.argpartition(dp, k)[:k]
        vals.append(len(set(nn_s) & set(nn_p)) / k)
    return float(np.mean(vals))

# ------------------
# 基于 similarity 的邻居聚合（只为算指标，不讲预测）
# ------------------

def neighbor_mean(D_sim, y, k=10):
    n = len(y)
    y_hat = np.zeros(n)
    for i in range(n):
        d = D_sim[i].copy()
        d[i] = np.inf
        nn = np.argpartition(d, k)[:k]
        y_hat[i] = np.mean(y[nn])
    return y_hat

# ------------------
# Task metrics
# ------------------

def mae(y, y_hat):
    return float(np.mean(np.abs(y - y_hat)))

def accuracy(y, y_hat):
    thr = np.median(y)
    return float(np.mean((y_hat <= thr) == (y <= thr)))

def precision_at_k(y, y_hat, k=10):
    true_top = set(np.argsort(y)[:k])
    pred_top = set(np.argsort(y_hat)[:k])
    return len(true_top & pred_top) / k

def ndcg_at_k(y, y_hat, k=10):
    order = np.argsort(y)
    gains = np.zeros(len(y))
    for i, idx in enumerate(order):
        gains[idx] = 1 / np.log2(i + 2)
    pred = np.argsort(y_hat)
    dcg = sum(gains[i] for i in pred[:k])
    idcg = sum(sorted(gains, reverse=True)[:k])
    return float(dcg / idcg)

# ------------------
# 主流程：三个任务一起
# ------------------

TASK_CONFIG = {
    "SlidingPuzzle": {
        "path": r"../processed_data/processed/per_task/SlidingPuzzle.pkl",
        "type": "sliding"
    },
    "Premarshalling": {
        "path": r"../processed_data/processed/per_task/Premarshalling.pkl",
        "type": "sliding"   # 排序任务，和 CVRP 用同一套指标
    },
    "CVRP": {
        "path": r"../processed_data/processed/per_task/CVRP.pkl",
        "type": "cvrp"
    },
    "BinPacking": {
        "path": r"../processed_data/processed/per_task/BinPacking.pkl",
        "type": "binpacking"
    }
}


for task, cfg in TASK_CONFIG.items():
    print(f"\n==== {task} ====")

    df = pd.read_pickle(cfg["path"])
    df = df.dropna(subset=["code", "objective"]).reset_index(drop=True)

    # AST parse_ok
    mask = df["code"].apply(ast_parse_ok)
    df = df[mask].reset_index(drop=True)
    print("AST parse_ok:", len(df))

    y = df["objective"].astype(float).values

    # Jaccard similarity matrix
    counters = [token_counter(c) for c in df["code"]]
    n = len(counters)
    D_sim = np.zeros((n, n), dtype=np.float32)

    for i in range(n):
        for j in range(i + 1, n):
            d = 1 - multiset_jaccard_from_counters(counters[i], counters[j])
            D_sim[i, j] = D_sim[j, i] = d

    D_perf = np.abs(y[:, None] - y[None, :])

    # Matrix correlation
    pr, pp = matrix_pearson(D_sim, D_perf)
    mr, mp = matrix_mantel(D_sim, D_perf)
    ov = overlap_at_k(D_sim, D_perf)

    print("Pearson:", pr)
    print("Mantel:", mr)
    print("Overlap@10:", ov)

    # Neighbor-based performance aggregation
    y_hat = neighbor_mean(D_sim, y)

    if cfg["type"] == "binpacking":
        print("MAE:", mae(y, y_hat))
        print("Accuracy:", accuracy(y, y_hat))

    elif cfg["type"] == "cvrp":
        print("Precision@10:", precision_at_k(y, y_hat))
        print("NDCG@10:", ndcg_at_k(y, y_hat))

    elif cfg["type"] == "sliding":
        print("MAE:", mae(y, y_hat))



==== SlidingPuzzle ====
AST parse_ok: 4858
Pearson: 0.03566213721382239
Mantel: 0.03566229
Overlap@10: 0.05413750514615068
MAE: 0.33097818402634827

==== Premarshalling ====
AST parse_ok: 4647
Pearson: 0.03351459750469627
Mantel: 0.033514548
Overlap@10: 0.06266408435549818
MAE: 2.054295132989025

==== CVRP ====
AST parse_ok: 1557
Pearson: 0.0588084974279175
Mantel: 0.058808506
Overlap@10: 0.01663455362877328
Precision@10: 0.0
NDCG@10: 0.39928778454700825

==== BinPacking ====
AST parse_ok: 4445
Pearson: 0.07844196006979202
Mantel: 0.07844205
Overlap@10: 0.021237345331833524
MAE: 0.29893860427446567
Accuracy: 0.6299212598425197
